# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
### API key management

### Reminder: Place .env file inside the root of the project folder so when calling the below from inside the notebook it should find the .env fule and load it inside the notebook environment
### PLEASE ADD THIS `.env` FILE TO YOUR PROJECT'S `.gitignore` file before committing and pushing the changes to your remote repo, as it contains API Keys and Secrets in it

import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

print("OPENAI_API_KEY" in os.environ)
print("LANGCHAIN_API_KEY" in os.environ)
print("TAVILY_API_KEY" in os.environ)


True
True
True


In [59]:
#import os
#import getpass

#os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [60]:
#os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [4]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related to mismanagement and improper handling by loan servicers. Specific issues include errors in loan balances, misapplication of payments, wrongful denials of payment plans, incorrect information on credit reports, difficulty in applying extra funds to principal, and transfers of loans without proper notification. Additionally, many complaints involve disputes over loan amounts, interest capitalization, and lack of transparency or communication from servicers.'

In [11]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, there are several complaints indicating that they were not handled in a timely manner. Specifically:\n\n- One complaint from 03/28/25 (Complaint ID: 12709087) was marked as "Not timely response?" = "No".\n- Another complaint from 04/24/25 (Complaint ID: 13160766) was handled "Yes" in a timely manner.\n- Multiple other complaints, such as those from 04/14/25, 04/18/25, and 05/02/25, also indicate timely responses, though their issues remain unresolved or ongoing.\n\nIn particular, the complaint with ID 12709087 explicitly states that the response was not timely. Furthermore, multiple complaints mention ongoing issues that have persisted for over a year, suggesting delays or failures in resolution.\n\nTherefore, yes, some complaints did not get handled in a timely manner.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to several interconnected reasons highlighted in the complaints:\n\n1. **Accumulating Interest and Lack of Transparency:** Many borrowers were unaware that interest continued to accumulate even during forbearance or deferment periods, leading to higher total debt over time. For example, some complaints mention that interest negated any payments made, and interest amounts were not clearly explained or understood by borrowers.\n\n2. **Limited Payment Options and Unaffordable Payments:** Borrowers were often limited to options like forbearance or deferment, which extended the repayment period and increased total debt. Increasing monthly payments to pay off loans faster was unaffordable for many, given their living expenses, job status, or income level.\n\n3. **Lack of Proper Communication and Notification:** Several complaints indicate that borrowers were not adequately notified about changes such as loan transfers, delinquency status, 

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers, including issues such as miscommunication, difficulty applying payments correctly, incorrect or bad information about the loan, and disputes over fees or terms. Many complaints involve frustrations with loan servicers not providing clear or accurate information, applying payments improperly, or failing to respond adequately.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, all the complaints listed indicate that the companies responded in a timely manner. Specifically, the complaints from rows 509, 288, 423, and 236 all show "Timely response?" marked as "Yes". Therefore, there are no complaints in the given data that were reported as not being handled in a timely manner.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often fail to pay back their loans due to issues such as miscommunication, lack of proper assistance, or administrative problems. For example, some individuals are unable to meet their repayment obligations because their payment plans are improperly handled—such as being steered into the wrong types of forbearances or having their autopayments inadvertently canceled without notification. Others encounter complications when their loans are transferred between servicers without clear communication, leading to missed payments or unawareness of their payment status. Additionally, some borrowers experience difficulties because they do not receive timely responses or assistance from their lenders or servicers when issues arise, such as unpaid bills or delays in processing deferments or forbearances. These administrative and communication failures can contribute significantly to loan repayment problems.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

#### ✅ Answer #1:
BM25 would be better when looking for very specific and unique text such as serial numbers, model numbers, error codes, policy document names, etc. This is because BM25 ranks results based on exact matches. Embedding-based retrievers prioritize semantic similarity and may not find unique values, especially if they lack semantic meaning.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers, including errors in loan balances, misapplied payments, wrongful denials of payment plans, and mishandling of loan information. Many complaints highlight issues such as incorrect or confusing loan data, lack of communication, unauthorized transfers of loans, and violations of privacy laws.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, it appears that some complaints were handled in a timely manner, as indicated by the responses labeled "Timely response? \'Yes\'". However, there are also complaints where the issue remained unresolved for an extended period, such as the case where the complainant has been waiting over a year or nearly 18 months without resolution. \n\nSpecifically, the complaint about the student loan account review has been open since an unspecified date ("XXXX") and has not been resolved after nearly 18 months, indicating a delay in handling that complaint. Similarly, the complaint about payments not appearing on the account remains unresolved over weeks.\n\nTherefore, yes, some complaints did not get handled in a timely manner, particularly those that have been ongoing for a year or more with no resolution.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Lack of Awareness: Many borrowers were unaware that they were required to repay their loans or were not properly informed about repayment obligations, especially when they first received financial aid.\n\n2. Poor Communication and Notifications: Borrowers experienced failure of lenders or servicers to notify them about due payments, changes in loan ownership, or the need to set up repayment plans.\n\n3. Difficulty Managing Payments: Borrowers found options like forbearance or deferment led to continued interest accumulation, which increased the total amount owed over time, making repayment more difficult.\n\n4. Accumulation of Interest: Even when payments were made, interest continued to accrue and often surpassed the payments made, resulting in rising balances and unmanageable debt.\n\n5. Financial Hardship and Unmanageable Conditions: Changes in economic circumstances, stagnant wages, and the inability to in

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be mismanagement by loan servicers including errors in loan balances, incorrect account classification, mishandling of payments, poor communication, and inadequate validation of loan information. Many borrowers report issues such as errors in reported balances, misapplied payments, inability to apply extra funds to principal, improper account updates, and delays or failures in correcting these errors. Additionally, problems related to unfair or confusing communication and mishandling of loan status during forbearance, consolidation, and forgiveness processes are prevalent.\n\nIn summary, the most common issue is **mismanagement and mishandling by loan servicers resulting in incorrect loan information, unapproved account actions, and poor customer service**.'

In [26]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints indicate that issues were not handled in a timely manner. Specifically:\n\n- The complaint with Complaint ID 12709087 (filed on 03/28/25) about problems with customer service was marked as "Timely response?": No, meaning it was not handled promptly.\n- The complaint with Complaint ID 12739706 (filed on 04/01/25) about improper reporting and account issues was also marked as "Timely response?": No.\n- Several other complaints from different companies, like Maximus Federal Services, Inc., have responses marked as "Yes" or "No," indicating whether the response was timely. Notably, some complaints about delays in corrections or responses were marked as "No" or highlighted delays exceeding acceptable timeframes.\n\nOverall, at least a few complaints clearly specify that they were not addressed in a timely manner, reflecting delays beyond standard or expected response times.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of systemic issues and mismanagement by loan servicers, as well as a lack of clear information about repayment options and the accumulation of interest. Specifically:\n\n- They were often steered into forbearance or deferment without proper explanation, leading to ongoing interest accrual that increased the total debt over time.\n- Many were not informed about income-driven repayment plans, loan forgiveness programs, or other legal options that could have made repayment more feasible.\n- The complexity and opacity of loan servicing systems, including improper transfers and insufficient communication, caused confusion and unintentional default or delinquency.\n- Some borrowed under questionable or predatory lending practices, with high interest rates and lack of transparency about the true cost of their loans.\n- In several cases, errors and misreporting by servicers, such as incorrect delinquency reporting and failur

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

#### 🎥✅ Answer #2:

User queries may not always be well formatted. Often, the hardest part of solving a problem is having a clear problem statement. For an analogy, think of an audience member asking a question of a presenter, and the presenter responding with: "I don't follow your question, could you state that another way, please". Having an LLM reformulate the questiion in different ways, increases the chances that the retriever will match documents that might have expressed the relevant idea with different wording.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to federal student loan servicing, such as errors in loan balances, misapplied payments, wrongful denials of payment plans, inaccurate reporting, and disputes over interest rates or debt legitimacy. Many complaints involve miscommunication, errors, or systemic breakdowns within the loan servicing system that significantly impact borrowers.\n\nIf you are asking for a specific, overarching issue, it seems that systemic errors and mismanagement by loan servicers are among the most prevalent problems reported.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all the complaints documented were responded to by the companies, but several explicitly state that they were handled "No" in terms of being timely. For example, complaints regarding loan processing delays and issues with contact or response times were not handled promptly. Specifically, the complaint involving Mohela\'s delays and lack of response to processing and customer service issues was marked as "Timely response?": "No." \n\nTherefore, yes, some complaints did not get handled in a timely manner.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors such as financial hardship, lack of proper information, and mismanagement by the loan servicers or colleges. For example, some borrowers experienced severe financial difficulties after graduation and relied on deferment or forbearance, which increased interest and made repayment more difficult. Others were misled by educational institutions regarding the value of their degrees and the long-term financial consequences, and were not properly informed about their repayment obligations. Additionally, issues like inability to establish payment plans, unnotified loan buyouts, and disputes over the legitimacy of debts also contributed to repayment failures.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints and issues related to loans, the most common issue appears to be related to errors in loan balances, misapplied or missing payments, and incorrect or inconsistent reporting of loan information. Many complaints highlight problems such as incorrect loan balances, uncredited payments, interest charges during in-school deferments, and inaccuracies in credit reports caused by loan servicer mistakes.\n\nAdditionally, issues with dealing with lenders or servicers—including inability to get clear information, problems with payment application, and mishandling of loan transfers—are prevalent.\n\nTherefore, the most common issue with loans, as evidenced by these complaints, is **errors and inaccuracies in loan balances and payment handling, often coupled with poor communication and misreporting by loan servicers.**'

In [39]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, yes, there are multiple complaints indicating that some complaints did not get handled in a timely manner. For example:\n\n- One complaint (Complaint ID: 12744910) regarding incorrect information on a report received a "Timely response" status of "Yes," but the complaint was about inaccuracies not being corrected promptly.\n- Another complaint (Complaint ID: 12935889) regarding servicing failure was marked as "No" for timely response, indicating it was not handled in a timely manner.\n- Several other complaints (e.g., Complaint ID: 12739706, 12709087) explicitly mention delays or failure to respond within certain timeframes.\n\nAdditionally, multiple complaints reference long wait times, lack of response, or unresolved issues after extended periods, which suggests that some complaints indeed were not handled promptly.\n\nIn summary, yes, there are complaints that were not handled in a timely manner according to the data.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often failed to pay back their loans due to a variety of systemic issues and mismanagement by loan servicers, including:\n\n1. Lack of proper notification or communication: Borrowers were not informed about when payments were due, changes in loan servicers, transfer of loan accounts, or options like income-driven repayment plans or forgiveness programs. For example, some reports indicated loans were transferred without notice, or borrowers were never informed that their payments needed to start.\n\n2. Errors and misreporting: There were instances where loans were incorrectly reported as delinquent or in default, sometimes after being in forbearance, leading to significant drops in credit scores and added financial hardship.\n\n3. Inability to access accurate information or documentation: Borrowers faced difficulties in obtaining clear statements, breakdowns of loan balances, and interest calculations. Many were unaware of accruing interest or the true amount owed, which often i

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [43]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related to loan servicing and communication. Many complaints involve difficulties in understanding or receiving accurate information about loan status, payments, and repayment terms; issues with auto-debit setups and discrepancies in payment processing; and concerns over improper or delayed responses from loan servicers. Additionally, issues such as improper reporting of delinquency, unauthorized account activity, and breaches of privacy are noted.\n\nIn summary, the most prevalent issues are:\n- Poor communication and lack of transparency from loan servicers.\n- Errors in payment processing and billing.\n- Disputes over loan account status and reporting.\n- Challenges in obtaining accurate and timely information.\n\nThese points suggest that the most common problem with loans is inadequate servicing and communication shortcomings.'

In [47]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that several complaints were marked as "Closed with explanation" and indicated responses such as "None" or "Closed with explanation," but the documentation suggests that some complaints were not handled in a timely manner or did not receive adequate responses. Specifically:\n\n- The complaint about Nelnet transferred account issues (Complaint ID: 13331376) was responded to as "Closed with explanation," despite ongoing serious issues and allegations of law violations.\n- Multiple complaints involving issues with payments and account errors were marked as "timely response: Yes," but the ongoing nature of the disputes and unresolved concerns imply that handling may have been insufficient or delayed from the complainants\' perspectives.\n\nTherefore, while responses were marked as timely in some cases, the nature of the complaints suggests that not all complaints were effectively or fully handled in a timely manner, especially considering the c

In [48]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including difficulties dealing with their lenders or servicers, administrative errors or delays, and disputes over the legitimacy or accuracy of the debt. Some individuals experienced issues such as receiving bad information about their loans, delays in re-amortization after forbearance ended, or problems with the reporting and collection of their debt, which led to defaults or incorrect account statuses. Additionally, there are cases where borrowers encountered alleged improper or illegal reporting, data breaches, or delays in processing payments, all of which contributed to their inability to fulfill repayment obligations.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

#### ✅ Answer #3:

Short answer is that semantic chunking in this case would group many of the short, similar sentences together. And, this would probably be desirable. But, if the bar is too low, it might group together sentences that are only superficially similar. To adjust this, you would raise the threshold, e.g. in this case, the percentile value that triggers a split.

I also wanted to make sure I understood the math and process behind semantic chunking. 


The Process
First, split document into sentences (langchain uses sentence-based splitter, e.g. nltk.sent_tokenizer)
Then do pairwise comparisons of the semantic distance between sentences.
Then group sentences if they are "close" in semantic meaning, and split to a new chunk when the semantic distance to the previous sentence exceeds the threshold. 

The Math

Percentile: 
Split when the semantic distance between two adjacent sentences is bigger than the n-th percentile of all adjacent-pair distances in the document. (Typical starting threshold: **75th or 80th percentile**. Lower the threshold to get smaller chunks (e.g. if topic shifts frequently) or raise the percentil to get fewer larger chunks)

Standard Deviation: 
Split when the semantic distance between two adjacent sentences exceeds the mean by nore than n times the stddev of all adjacent-pair distances in the document. (Typical value: **n = 1.5**; threshold = **Mean + 1.5 × StdDev**)

Interquartile:
Split when the distance falls above the Q3 percentile by some factor, "n", of the "interquartile range" (IQR). IQR is the Q3 percentile - Q1 percentile. Example: threshold = Q3 + n x IQR (Typical value: **n = 1.5**; threshold = **Q3 + 1.5 × IQR**)

Gradient:
This is kind of like a first order derivative, compared to the others. It essentially measures the velocity of change between the sentences (as opposed ot the distance), triggering a split when there is a sudden change. To do this, after calculating the distances between each set of pairs, you then calculate the gradients (the distances between the distances). The rule would be to split whenever the gradient exceeds a threshold, **N**.  
(Typical value: **N = 0.10**)

🎥 See example [here](semantic_chunking_examples.md) that I worked out with ChatGPT



# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [49]:
### YOUR CODE HERE





#### Study Note
these are my notes taken during breakout room:

generate golde dataest
naive, bm25, reranking, etc.
should find all the code from 8
import ragas llms from 7 
#uv add ragas 
#nltk?
from raqas.llms import Langchainllmwrapper 

#### Study Note: Documentation
I'm going to document the heck out of this.
If I track every step that I do, I'll know what to say in my Loom video 🤞
First off, I'm eager and nervous to try this.
I'm pretty sure I can find all the code I need in notebooks from session 7 & 8

#### Study Note: Design Idea
Activity 1 instructions seem pretty clear, but I ran it by ChatGPT to confirm.
I need to 
1. Create synth dataset for test case
2. Evaluate 5 retreival approaches: Naive, BM25, Multi-Query, Parent Doc, Ensemble with Ragas
    a. run each retriever with the golden dataset
    b. evaluate each run with Ragas
3. Then I'll have to do something with LangSmith to get the latency and cost, but I'll think about that later



## Task 1: Create a dataset

I will find the ragas code from notebook 7

First, I'll do the nltk thing. I still don't undertsand it, but the comment in 7 said it would prevent (mac related) os errors.
It worked there, so I will keep it.
First stumbling block: it failed. But, I figured out why pretty quick. This notebook didn't install nltk

#### Study Note: Environment setup
Till now, I've just been relying on uv sync, and not thinking much about dependencies.
So, I took a look at the project.toml for this notebook vs 7. Lots of stuff in 7 that isn't here, and I think I'm going to need it.
But, I don't want to break anything, so, I'll just add things as I run into needing them (starting with nltk). Also, I'll just install via terminal first, and if nothing breaks, I'll update the toml file later.
Clearly, I'm also going to need ragas
Ran this command:  uv pip install nltk ragas==0.2.10



#### Study Notes: To-dos
Track thinkgs here that are pending
update toml with nltk and ragas (see above)
update toml with rapidfuzz (see below)

In [50]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/family/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/family/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

That worked. Baby steps!

I'm going to end up doing something with LangSmith, so I'll steal the next cell from earlier notebooks (plus a print statement)

In [ ]:
os.environ["LANGCHAIN_PROJECT"] = "Number9 Evaluation"
os.environ["LANGCHAIN_TRACING_V2"] = "true"


Now I'll import stuff and set the llm 


In [52]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Taking the 'abstracted SDG' from HW7
I ran it just as it was in HW7, and I got an error, because there was no such thing as "docs", so i figured out that I needed to put in "loan_complaint_data"
Pretty happy that I figured that out, with only a slight nudge from ChatGPT
It failed, with a nice clear error message that I needed to install "rapidfuzz".

Don't know why I need rapidfuzz now, when we didn't need it in HW7. 
I suppose it is because of the structure of the data.
ChatGPT gave me some other hogwash that I don't believe. I'm sticking with my guess, and it doesn't really matter.
I installed rapidfuzz (in terminal window; I'll need to add a cell or put it in the toml file later)<< added to toml!

In [80]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
golden_dataset = generator.generate_with_langchain_docs(loan_complaint_data[:20], testset_size=10)

Applying SummaryExtractor:   0%|          | 0/14 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/20 [00:00<?, ?it/s]

Node 2bd224e2-0c03-4fa7-ac80-98135d92cd3d does not have a summary. Skipping filtering.
Node b4a321cf-5344-4b04-9c2b-60f200312ea0 does not have a summary. Skipping filtering.
Node 73763f28-5e3e-41ff-aed0-d82f74a36580 does not have a summary. Skipping filtering.
Node 53ecc392-5478-43c6-a987-15627ef13a11 does not have a summary. Skipping filtering.
Node ab248320-233d-4a08-9346-0daa3328ed80 does not have a summary. Skipping filtering.
Node a29db1db-9214-43a3-9f2a-35d762605a8d does not have a summary. Skipping filtering.


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/51 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

Wow, that worked!
Now, I need to see the dataset, so I'm stealing the next cell from notebook 7

I am gobsmacked that I got this working in under an hour (not counting annotations) Insert excessive emojis here: ✅❤️🥳🎉
Time to (commit the code), take a victory lap, and call it a night.

In [ ]:
golden_dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,When did the federal student loan COVID-19 for...,[The federal student loan COVID-19 forbearance...,The federal student loan COVID-19 forbearance ...,single_hop_specifc_query_synthesizer
1,What are the legal implications for Aidvantage...,[I submitted my annual Income-Driven Repayment...,The context indicates that Aidvantage has assi...,single_hop_specifc_query_synthesizer
2,Did my data breach violate FERPA and can I get...,[My personal and financial data was compromise...,My personal and financial data was compromised...,single_hop_specifc_query_synthesizer
3,What is the issue with the information provide...,"[According to Studentaid.gov, Im to get an ema...","According to the context, Studentaid.gov state...",single_hop_specifc_query_synthesizer
4,How does the Fair Credit Reporting Act ensure ...,[I am writing to formally dispute inaccurate i...,"The Fair Credit Reporting Act (FCRA), specific...",single_hop_specifc_query_synthesizer
5,Did Department of Education get involved with ...,[<1-hop>\n\nBreach of Contract - All four bran...,"Yes, the Department of Education was involved ...",multi_hop_specific_query_synthesizer
6,How does the illegal access to the Department ...,"[<1-hop>\n\nOn XXXX XXXX XXXX, XXXX XXXX instr...",The illegal access to the Department of Educat...,multi_hop_specific_query_synthesizer
7,Based on the issues raised with NelNet's handl...,[<1-hop>\n\nI keep setting up auto-debit with ...,The improper management of auto-debit records ...,multi_hop_specific_query_synthesizer
8,How does the continued reporting and collectio...,[<1-hop>\n\nThis account was transferred to Ne...,The continued reporting and collection of stud...,multi_hop_specific_query_synthesizer
9,How does the violation of federal privacy laws...,[<1-hop>\n\nThis is a formal legal demand for ...,The violation of federal privacy laws by the D...,multi_hop_specific_query_synthesizer


Coming back to work on this, after completing Hw10, so now I have to reset **my** context.
I went ahead and added nltk and ragas to project.toml, so I can clean up my 'technical debt'
Reran the notebook, and it all worked.
Now, I ended up with 10 new questions - 

Going to stash this dataset into a json, just in case I need it back later

In [93]:
import json

with open("goldendataset.json", "w") as f:
    json.dump([sample.model_dump() for sample in golden_dataset], f, indent=2)


some code for later, just in case I do need to get back then golden_data from the json file

In [ ]:
# from ragas.testset import SingleTurnTestset

# with open("golden_dataset.json", "r") as f:
#     data = json.load(f)

# golden_dataset = SingleTurnTestset.model_validate(data)


## Task 2
Evaluate 5 retreival approaches: Naive, BM25, Multi-Query, Parent Doc, Ensemble with Ragas

### Task 2.1 Run retreivers

Next step: Run a retriever with the golden data set
Starting slowly, with one retriever and saving the outputs

here are the names of the retrieval chains, from above, for my reference
naive_retrieval_chain
bm25_retrieval_chain
contextual_compression_retrieval_chain
multi_query_retrieval_chain
parent_document_retrieval_chain
ensemble_retrieval_chain


first, just try one retriever

In [ ]:
#first I'll try to run, with BM25 (hard-coded)
def run_retriever_on_dataset(retriever_chain, golden_dataset): #since this version is hard-coded, it never actually uses the retriever_chain parameter
    outputs = []

    for test_row in golden_dataset:
        response = bm25_retrieval_chain.invoke({
            "question": test_row.eval_sample.user_input})
        outputs.append({
            "user_input": test_row.eval_sample.user_input,
            "reference": test_row.eval_sample.reference,
            "response": response["response"].content if hasattr(response["response"], "content") else response["response"],
            "retrieved_contexts": [ctx.page_content for ctx in response["context"]],
        })

    return outputs

bm25_outputs = run_retriever_on_dataset(bm25_retrieval_chain, golden_dataset)


In [84]:
import pandas as pd

bm25_df = pd.DataFrame(bm25_outputs)
bm25_df


,question,reference,response,contexts
0,Did the COVID-19 forbearance program end befor...,The federal student loan COVID-19 forbearance ...,"Based on the provided context, the COVID-19 fo...",[The federal student loan COVID-19 forbearance...
1,What is the significance of the SAVE Plan in r...,The context discusses a borrower’s situation w...,The significance of the SAVE Plan in relation ...,[I applied for and was officially approved und...
2,FERPA is violated what I do?,My personal and financial data was compromised...,If FERPA (Family Educational Rights and Privac...,"[I signed a loan through XXXX, I paid my loans..."
3,Is 15 U.S.C. 1681i the law that requires credi...,"Yes, 15 U.S.C. 1681i is part of the Fair Credi...","Yes, 15 U.S.C. 1681i is the law that requires ...",[I am writing to formally dispute inaccurate i...
4,Can you explain what Aid Avantage is and how i...,Aid Avantage is mentioned in the context as pa...,Aid Avantage is a student loan servicer or rel...,[I am devastated. I would like to report a sit...
5,Considering the issues with NelNet's handling ...,A legal compliance officer should recommend th...,Given the issues with NelNet's handling of aut...,[I am writing to formally express my concerns ...
6,Considering the breach of personal information...,The context indicates that XXXX XXXX XXXX inst...,To ensure compliance with federal privacy laws...,"[In XX/XX/XXXX, reports confirmed that the Dep..."
7,How does the illegal access by DOGE relate to ...,"The context describes how DOGE, an entity unde...",The illegal access by DOGE (Department of Gove...,"[On XX/XX/XXXX, XXXX XXXX instructed his staff..."
8,How does the FCRA address the requirement for ...,The FCRA mandates that credit reporting agenci...,The Fair Credit Reporting Act (FCRA) requires ...,[XXXX XXXX [ XXXX XXXX XXXXXXXX XXXX XXXX ] [...
9,How did NelNet's handling of automatic payment...,The individual experienced repeated issues wit...,NelNet's handling of automatic payments and th...,[The company NelNet did not have the correct c...


Now, convert output to a pandas.DataFrame first, and then into a Ragas EvaluationDataset:

In [91]:
import pandas as pd
from ragas import EvaluationDataset

# Step 1: Convert to DataFrame
bm25_df = pd.DataFrame(bm25_outputs)

# Step 2: Convert to Ragas-compatible EvaluationDataset
bm25_eval_dataset = EvaluationDataset.from_pandas(bm25_df)


### Evaluate first retriever (BM25)
borrowing some more code fron HW8, for the eval:

In [77]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

In [92]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=bm25_eval_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Exception raised in Job[25]: AttributeError('StringIO' object has no attribute 'statements')
Exception raised in Job[35]: APIConnectionError(Connection error.)


{'context_recall': 0.8417, 'faithfulness': 0.7594, 'factual_correctness': 0.5620, 'answer_relevancy': 0.6619, 'context_entity_recall': 0.4776, 'noise_sensitivity_relevant': 0.2619}

### Party time!

I ran an evaluation for 1 retriever; time for another victory lap. 
first, stash the output for safe-keeping (since it takes 6 minutes), then figure out how to externalize the retriever name so I can loop through them all

BM25 output from first run:
{'context_recall': 0.8417, 'faithfulness': 0.7594, 'factual_correctness': 0.5620, 'answer_relevancy': 0.6619, 'context_entity_recall': 0.4776, 'noise_sensitivity_relevant': 0.2619}

instead of updating the run_retriever_on_dataset definition from above, I'll make a newer copy below, just to continue to trace my process

Define retriever names, to use with the retriever

In [ ]:
retrievers = {
    #"bm25": bm25_retrieval_chain, commented out because it was already run
    "naive": naive_retrieval_chain,
    "multi_query": multi_query_retrieval_chain,
    "parent_doc": parent_document_retrieval_chain,
    "ensemble": ensemble_retrieval_chain,
    "contextual_compression": contextual_compression_retrieval_chain,
}


In [94]:
#updated version of the code, with the retriever name externalized

def run_retriever_on_dataset(name, retriever_chain, golden_dataset):
    print(f"Running {name} on golden dataset")
    outputs = []

    for test_row in golden_dataset:
        response = retriever_chain.invoke({
            "question": test_row.eval_sample.user_input})
        outputs.append({
            "user_input": test_row.eval_sample.user_input,
            "reference": test_row.eval_sample.reference,
            "response": response["response"].content if hasattr(response["response"], "content") else response["response"],
            "retrieved_contexts": [ctx.page_content for ctx in response["context"]],
            "retriever_name": name #this is the new addition for being able to keep track of the retriever name later
        })

    return outputs



next cell looked like a good idea, but it was overengineerd and didn't give what I wanted / needed for output format
something about dictionary of lists vs list of dictionaries 
and anyway, I would have had to split up the output because I don't want to run 40 minutes worth of evals in one cell
this tends to happen when I listen to the AI too much

In [ ]:

# all_outputs = {} #initializing the dictionary to store the outputs

# for name, chain in retrievers.items():
#     print(f"Running: {name}")
#     outputs = run_retriever_on_dataset(name, chain, golden_dataset)
#     all_outputs[name] = outputs #adding the outputs to a single dictionary



Running: naive
Running naive on golden dataset
Running: multi_query
Running multi_query on golden dataset
Running: parent_doc
Running parent_doc on golden dataset
Running: ensemble
Running ensemble on golden dataset


this is the brute force version, with each retriever run separately, which works for me especially since I might need to run some of the retrievers individually, or multiple times
which I will now need to do, because contextual compression failed with a rate limit error. I will retry with Cohere production key.
(Production key worked 👍)

In [104]:
# bm25_outputs = run_retriever_on_dataset("bm25", bm25_retrieval_chain, golden_dataset) #this was the first version, hard-coded
# naive_outputs = run_retriever_on_dataset("naive", naive_retrieval_chain, golden_dataset)
# multi_query_outputs = run_retriever_on_dataset("multi_query", multi_query_retrieval_chain, golden_dataset)
# parent_doc_outputs = run_retriever_on_dataset("parent_doc", parent_document_retrieval_chain, golden_dataset)
# ensemble_outputs = run_retriever_on_dataset("ensemble", ensemble_retrieval_chain, golden_dataset)
contextual_compression_outputs = run_retriever_on_dataset("contextual_compression", contextual_compression_retrieval_chain, golden_dataset)

Running contextual_compression on golden dataset


I want to do some validation. 
Next code cell is straiht from ChatGPT. 
this is the kind of stuff it is pretty good at, and I don't feel like I need to dive into the details

In [105]:
expected_keys = {"user_input", "reference", "response", "retrieved_contexts"}

for name, output in [
    ("bm25", bm25_outputs),
    ("naive", naive_outputs),
    ("multi_query", multi_query_outputs),
    ("parent_doc", parent_doc_outputs),
    ("ensemble", ensemble_outputs),
    ("contextual_compression", contextual_compression_outputs),
]:
    if not isinstance(output, list):
        print(f"{name}: ❌ Not a list")
        continue

    if not output:
        print(f"{name}: ⚠️ Empty list")
        continue

    sample = output[0]
    if not isinstance(sample, dict):
        print(f"{name}: ❌ First item is not a dict")
        continue

    missing = expected_keys - set(sample.keys())
    if missing:
        print(f"{name}: ❌ Missing keys: {missing}")
    else:
        print(f"{name}: ✅ Format looks good ({len(output)} samples)")


bm25: ✅ Format looks good (10 samples)
naive: ✅ Format looks good (10 samples)
multi_query: ✅ Format looks good (10 samples)
parent_doc: ✅ Format looks good (10 samples)
ensemble: ✅ Format looks good (10 samples)
contextual_compression: ✅ Format looks good (10 samples)


As before, converting the outputs to evaluation datasets. Probably there's a more efficient approach

In [106]:
import pandas as pd
from ragas import EvaluationDataset

# Step 1: Convert to DataFrame
# bm25_df = pd.DataFrame(bm25_outputs)
naive_df = pd.DataFrame(naive_outputs)
multi_query_df = pd.DataFrame(multi_query_outputs)
parent_doc_df = pd.DataFrame(parent_doc_outputs)
ensemble_df = pd.DataFrame(ensemble_outputs)
contextual_compression_df = pd.DataFrame(contextual_compression_outputs)

# Step 2: Convert to Ragas-compatible EvaluationDataset
# bm25_eval_dataset = EvaluationDataset.from_pandas(bm25_df)
naive_eval_dataset = EvaluationDataset.from_pandas(naive_df)
multi_query_eval_dataset = EvaluationDataset.from_pandas(multi_query_df)
parent_doc_eval_dataset = EvaluationDataset.from_pandas(parent_doc_df)
ensemble_eval_dataset = EvaluationDataset.from_pandas(ensemble_df)
contextual_compression_eval_dataset = EvaluationDataset.from_pandas(contextual_compression_df)



putting all the ragas imports together, here:

In [ ]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

Next cell is same as I used above for BM25, but now putting it into a loop to run all the evals together
hoping this:

results[name] = result give me a separate, named, result set for each retriever 🤞

In [109]:
custom_run_config = RunConfig(timeout=600)
results = {}

datasets = [
    ("bm25", bm25_eval_dataset),
    ("naive", naive_eval_dataset),
    ("multi_query", multi_query_eval_dataset),
    ("parent_doc", parent_doc_eval_dataset),
    ("ensemble", ensemble_eval_dataset),
    ("contextual_compression", contextual_compression_eval_dataset)
]

for name, dataset in datasets:
    print(f"Evaluating: {name}")
    try:
        result = evaluate(
            dataset=dataset,
            metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
            llm=evaluator_llm,
            run_config=custom_run_config
        )
        results[name] = result
    except Exception as e:
        print(f"❌ Error during {name}: {e}")
        results[name] = None  # or skip entirely


Evaluating: bm25


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Evaluating: naive


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Exception raised in Job[35]: APIConnectionError(Connection error.)
Exception raised in Job[41]: TimeoutError()
Exception raised in Job[59]: TimeoutError()


...

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Exception raised in Job[22]: APIConnectionError(Connection error.)
Exception raised in Job[31]: APIConnectionError(Connection error.)
Exception raised in Job[37]: APIConnectionError(Connection error.)
Exception raised in Job[35]: APIConnectionError(Connection error.)
Exception raised in Job[52]: APIConnectionError(Connection error.)
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[41]: TimeoutError()
Exception raised in Job[47]: TimeoutError()
Exception raised in Job[53]: TimeoutError()
Exception raised in Job[59]: TimeoutError()


Evaluating: contextual_compression


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Exception raised in Job[56]: AttributeError('StringIO' object has no attribute 'statements')
Exception raised in Job[35]: APIConnectionError(Connection error.)


### Party time again!
The evals all ran. took 56 minutes. Maybe I shouldn't have done them all at once?
I had to increase the timeout; I was getting too many errors at 360, so upped it to 10 minutes.
Ensemble and multi-query still generated too many timeout errors

Run the next cell to make sure I really got the results, the way I expected

In [111]:
results["contextual_compression"]


{'context_recall': 0.7583, 'faithfulness': 0.6673, 'factual_correctness': 0.5311, 'answer_relevancy': 0.8500, 'context_entity_recall': 0.5427, 'noise_sensitivity_relevant': 0.2390}

That worked, so lets try looping them all

In [113]:
for name in results:
    print(f"\n{name.upper()} RESULTS:")
    print(results[name])



BM25 RESULTS:
{'context_recall': 0.8667, 'faithfulness': 0.8118, 'factual_correctness': 0.5210, 'answer_relevancy': 0.7582, 'context_entity_recall': 0.4540, 'noise_sensitivity_relevant': 0.2374}

NAIVE RESULTS:
{'context_recall': 0.8417, 'faithfulness': 0.7532, 'factual_correctness': 0.5370, 'answer_relevancy': 0.9424, 'context_entity_recall': 0.4445, 'noise_sensitivity_relevant': 0.2966}

MULTI_QUERY RESULTS:
{'context_recall': 0.8667, 'faithfulness': 0.8165, 'factual_correctness': 0.4462, 'answer_relevancy': 0.7477, 'context_entity_recall': 0.4583, 'noise_sensitivity_relevant': 0.3406}

PARENT_DOC RESULTS:
{'context_recall': 0.8083, 'faithfulness': 0.8284, 'factual_correctness': 0.5050, 'answer_relevancy': 0.7498, 'context_entity_recall': 0.3631, 'noise_sensitivity_relevant': 0.2905}

ENSEMBLE RESULTS:
{'context_recall': 0.9417, 'faithfulness': 0.8553, 'factual_correctness': 0.5030, 'answer_relevancy': 0.9399, 'context_entity_recall': 0.4396, 'noise_sensitivity_relevant': 0.2475}

C

Got all the results; sav

### Success
I got the Ragas results! I saved them to rags_results_1.
I also need cost and latency, which I can get from LangGr

## Step 3: Get latency and cost
Now I need to use Langchain.
Turns out, I should have thought of this sooner, because I probably could have used the same retriever runs to trace with LangChain and also use as output for Ragas. Oh, well, they don't take so long to run. 

Main thing is I have to figure out how to configure tracing. I'll root around in homework 7, and then bug one of the AI assistants if that doesn't work


Langchain setup (tried to do this above, but let's get everything in one place and get it working)

In [133]:
os.environ["LANGCHAIN_PROJECT"] = "Number9 Evaluation"
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [ ]:
eval_llm = ChatOpenAI(model="gpt-4.1")

In [ ]:
#magic France test cell
from langchain_openai import ChatOpenAI
from langchain_core.tracers import LangChainTracer
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define components
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
prompt = ChatPromptTemplate.from_template("What is the most common issue with loans?")
chain = prompt | llm | StrOutputParser()

# Attach a tracer manually (failsafe if env vars don't take)
tracer = LangChainTracer()
chain_with_tracing = chain.with_config({"callbacks": [tracer]})

# Invoke chain
response = chain_with_tracing.invoke({})
print(response)


The most common issue with loans is the inability of borrowers to make timely payments, leading to default on the loan. This can be due to various reasons such as financial difficulties, unexpected expenses, or poor financial management. Defaulting on a loan can have serious consequences such as damage to credit score, additional fees and penalties, and potential legal action by the lender.


In [134]:
bm25_outputs_traced = run_retriever_on_dataset("bm25", bm25_retrieval_chain, golden_dataset)


Running bm25 on golden dataset


In [123]:
# Step 1: Verify your environment variables are set
import os
from dotenv import load_dotenv

# Make sure your .env file is loaded
load_dotenv(dotenv_path="../.env")

# Check if all required keys are present
required_keys = ["OPENAI_API_KEY", "LANGCHAIN_API_KEY", "LANGCHAIN_TRACING_V2"]
for key in required_keys:
    if key in os.environ:
        print(f"✅ {key}: SET")
    else:
        print(f"❌ {key}: MISSING")

# Set the project name for organizing traces
os.environ["LANGCHAIN_PROJECT"] = "Advanced-Retrieval-Evaluation"
os.environ["LANGCHAIN_TRACING_V2"] = "true"

print(f"\n🎯 LangSmith Project: {os.environ.get('LANGCHAIN_PROJECT')}")
print(f"📊 Tracing Enabled: {os.environ.get('LANGCHAIN_TRACING_V2')}")


✅ OPENAI_API_KEY: SET
✅ LANGCHAIN_API_KEY: SET
✅ LANGCHAIN_TRACING_V2: SET

🎯 LangSmith Project: Advanced-Retrieval-Evaluation
📊 Tracing Enabled: true


In [125]:
# 🔍 Comprehensive LangSmith Diagnostic
import os
from dotenv import load_dotenv

print("🔧 LANGSMITH DIAGNOSTIC REPORT")
print("=" * 50)

# 1. Check environment variables
print("\n1️⃣ ENVIRONMENT VARIABLES:")
load_dotenv(dotenv_path="../.env")

env_vars = {
    "LANGCHAIN_API_KEY": os.environ.get("LANGCHAIN_API_KEY", "NOT SET"),
    "LANGCHAIN_TRACING_V2": os.environ.get("LANGCHAIN_TRACING_V2", "NOT SET"),
    "LANGCHAIN_PROJECT": os.environ.get("LANGCHAIN_PROJECT", "NOT SET"),
    "LANGCHAIN_ENDPOINT": os.environ.get("LANGCHAIN_ENDPOINT", "NOT SET")
}

for key, value in env_vars.items():
    if value == "NOT SET":
        print(f"❌ {key}: {value}")
    elif key == "LANGCHAIN_API_KEY":
        print(f"✅ {key}: {value[:8]}...{value[-4:] if len(value) > 12 else 'TOO SHORT'}")
    else:
        print(f"✅ {key}: {value}")

# 2. Force set the environment variables again
print("\n2️⃣ FORCE SETTING VARIABLES:")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Advanced-Retrieval-Debug"
print(f"✅ Set LANGCHAIN_TRACING_V2 = {os.environ['LANGCHAIN_TRACING_V2']}")
print(f"✅ Set LANGCHAIN_PROJECT = {os.environ['LANGCHAIN_PROJECT']}")

# 3. Test LangSmith client directly
print("\n3️⃣ TESTING LANGSMITH CLIENT:")
try:
    from langsmith import Client
    client = Client()
    print(f"✅ LangSmith Client created successfully")
    print(f"✅ API URL: {client.api_url}")
    
    # Try to list projects to test connection
    projects = list(client.list_projects(limit=5))
    print(f"✅ Can connect to LangSmith - found {len(projects)} projects")
    
except Exception as e:
    print(f"❌ LangSmith Client failed: {e}")

print("\n" + "=" * 50)


🔧 LANGSMITH DIAGNOSTIC REPORT

1️⃣ ENVIRONMENT VARIABLES:
✅ LANGCHAIN_API_KEY: lsv2_pt_...7320
✅ LANGCHAIN_TRACING_V2: true
✅ LANGCHAIN_PROJECT: Advanced-Retrieval-Evaluation
❌ LANGCHAIN_ENDPOINT: NOT SET

2️⃣ FORCE SETTING VARIABLES:
✅ Set LANGCHAIN_TRACING_V2 = true
✅ Set LANGCHAIN_PROJECT = Advanced-Retrieval-Debug

3️⃣ TESTING LANGSMITH CLIENT:
✅ LangSmith Client created successfully
✅ API URL: https://api.smith.langchain.com
✅ Can connect to LangSmith - found 5 projects



In [126]:
# 🔧 FIX THE ENDPOINT ISSUE
import os

print("🔧 FIXING LANGSMITH ENDPOINT")
print("=" * 40)

# Set the correct LangSmith endpoint
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Advanced-Retrieval-Fixed"

print("✅ Set LANGCHAIN_ENDPOINT =", os.environ["LANGCHAIN_ENDPOINT"])
print("✅ Set LANGCHAIN_TRACING_V2 =", os.environ["LANGCHAIN_TRACING_V2"])
print("✅ Set LANGCHAIN_PROJECT =", os.environ["LANGCHAIN_PROJECT"])

# Quick test again
print("\n🧪 Testing with fixed endpoint...")
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
prompt = ChatPromptTemplate.from_template("Say 'Endpoint fixed test' and the time {time}")
chain = prompt | llm

import time
result = chain.invoke({"time": time.strftime("%H:%M:%S")})

print("✅ Test completed!")
print("🔗 Check LangSmith for project: Advanced-Retrieval-Fixed")
print("   Should appear in 1-2 minutes")
print("=" * 40)


🔧 FIXING LANGSMITH ENDPOINT
✅ Set LANGCHAIN_ENDPOINT = https://api.smith.langchain.com
✅ Set LANGCHAIN_TRACING_V2 = true
✅ Set LANGCHAIN_PROJECT = Advanced-Retrieval-Fixed

🧪 Testing with fixed endpoint...
✅ Test completed!
🔗 Check LangSmith for project: Advanced-Retrieval-Fixed
   Should appear in 1-2 minutes


In [135]:
# 🔧 SIMPLE FIX: Run chains directly like your working test
import os
import time

print("🎯 LangSmith tracing setup (like your working test):")
os.environ["LANGCHAIN_TRACING_V2"] = "true"  # This worked before!

# Test with one simple question (like your working France test)
test_question = "What is the most common issue with loans?"

print(f"\n🧪 Testing each retriever with: '{test_question}'")
print("📊 Measuring latency and sending to LangSmith default project...")

# Store timing results
latency_results = {}

retrievers_to_test = {
    "bm25": bm25_retrieval_chain,
    # "naive": naive_retrieval_chain, 
    # "multi_query": multi_query_retrieval_chain,
    # "parent_doc": parent_document_retrieval_chain,
    # "ensemble": ensemble_retrieval_chain,
    # "contextual_compression": contextual_compression_retrieval_chain
}

for name, chain in retrievers_to_test.items():
    print(f"\n🔄 Testing {name}...")
    
    start_time = time.time()
    try:
        # Run directly like your working test - this SHOULD show in LangSmith
        result = chain.invoke({"question": test_question})
        end_time = time.time()
        
        latency = end_time - start_time
        latency_results[name] = latency
        
        print(f"  ✅ Success: {latency:.2f}s")
        print(f"  📝 Response preview: {result['response'].content[:100]}...")
        
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        latency_results[name] = None

print(f"\n📊 LATENCY SUMMARY:")
for name, latency in latency_results.items():
    if latency:
        print(f"  {name}: {latency:.2f}s")
    else:
        print(f"  {name}: FAILED")

print(f"\n🔗 Check your LangSmith 'default' project for 6 new traces!")
print("   Each trace should show cost/token data")
print("=" * 60)

🎯 LangSmith tracing setup (like your working test):

🧪 Testing each retriever with: 'What is the most common issue with loans?'
📊 Measuring latency and sending to LangSmith default project...

🔄 Testing bm25...
  ✅ Success: 0.93s
  📝 Response preview: Based on the provided context, the most common issue with loans appears to be problems related to de...

📊 LATENCY SUMMARY:
  bm25: 0.93s

🔗 Check your LangSmith 'default' project for 6 new traces!
   Each trace should show cost/token data


In [142]:
import os
import time

# Use EXACTLY the same setup as your working France test
os.environ["LANGCHAIN_TRACING_V2"] = "true"

print("🧪 Using EXACT same pattern as working France test...")

# Create components exactly like France test
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

test_llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
test_prompt = ChatPromptTemplate.from_template("Answer this question: {question}")
test_chain = test_prompt | test_llm

# Test with your loan question
question = "What is the most common issue with loans?"
print(f"Testing with: {question}")

start_time = time.time()
result = test_chain.invoke({"question": question})
end_time = time.time()

print(f"✅ Test completed in {end_time - start_time:.2f} seconds")
print(f"📝 Result: {result.content}")
print("\n🔗 Check LangSmith - this should appear like France did!")

🧪 Using EXACT same pattern as working France test...
Testing with: What is the most common issue with loans?
✅ Test completed in 0.86 seconds
📝 Result: The most common issue with loans is the risk of default, where borrowers are unable to repay the borrowed amount as agreed. This can lead to financial losses for lenders and can be caused by factors such as economic downturns, poor creditworthiness of borrowers, or unexpected financial hardships. Other common issues include high interest rates, hidden fees, and inadequate understanding of loan terms.

🔗 Check LangSmith - this should appear like France did!
